# 🧹 Notebook 02: EDA & Preprocessing Data
## Analitika Data Keuangan Sektor Publik

**Tujuan Pembelajaran:**
1. Melakukan Exploratory Data Analysis (EDA) secara menyeluruh
2. Mendeteksi dan menangani *missing values*
3. Mendeteksi dan menangani *outlier*
4. Melakukan transformasi data (normalisasi, encoding)
5. Menyiapkan dataset yang bersih untuk analisis lanjutan

---
> **Dataset:** `keuangan_pemda.csv` — Dataset ini **sengaja mengandung** missing values, nilai negatif, dan outlier untuk tujuan latihan preprocessing.

## 📦 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:,.0f}')

print('Library berhasil dimuat ✅')

## 📂 2. Memuat Dataset

In [ ]:
df = pd.read_csv('../../Dataset/02-EDA/keuangan_pemda.csv')
df_original = df.copy()  # Simpan salinan original

print(f'Dataset dimuat: {df.shape[0]} baris × {df.shape[1]} kolom')
df.head(10)

## 🔍 3. Phase 1: Data Understanding (EDA)

In [ ]:
# 3.1 Informasi umum dataset
print('=' * 55)
print('RINGKASAN DATASET')
print('=' * 55)
print(f'  Jumlah baris   : {df.shape[0]}')
print(f'  Jumlah kolom   : {df.shape[1]}')
print(f'  Ukuran dataset : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print()
df.info()

In [ ]:
# 3.2 Statistik deskriptif
print('STATISTIK DESKRIPTIF (nilai dalam Rupiah)')
df.describe()

In [ ]:
# 3.3 Audit missing values secara komprehensif
print('AUDIT MISSING VALUES')
print('=' * 45)

missing_info = pd.DataFrame({
    'Jumlah Missing': df.isnull().sum(),
    'Persentase (%)': (df.isnull().sum() / len(df) * 100).round(2),
    'Tipe Data': df.dtypes
})
missing_info = missing_info[missing_info['Jumlah Missing'] > 0]

if len(missing_info) > 0:
    print(missing_info)
else:
    print('Tidak ada missing values!')

# Visualisasi
fig, ax = plt.subplots(figsize=(10, 4))
missing_pct = (df.isnull().sum() / len(df) * 100)
missing_pct = missing_pct[missing_pct > 0]
if len(missing_pct) > 0:
    bars = ax.barh(missing_pct.index, missing_pct.values, color='#e74c3c', edgecolor='white')
    ax.set_xlabel('Persentase Missing (%)')
    ax.set_title('Missing Values per Kolom', fontweight='bold')
    for bar, val in zip(bars, missing_pct.values):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center')
    plt.tight_layout()
    plt.show()

In [ ]:
# 3.4 Deteksi nilai tidak wajar (negatif)
print('DETEKSI NILAI NEGATIF PADA KOLOM NUMERIK')
print('=' * 50)

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

for col in num_cols:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f'  ⚠️  {col}: {neg_count} nilai negatif')
        print(df[df[col] < 0][[col]].head().to_string())
        print()

## 📈 4. Phase 2: Visualisasi EDA

In [ ]:
# 4.1 Boxplot untuk mendeteksi outlier pada kolom utama
cols_to_plot = ['Anggaran_APBD', 'Realisasi_APBD', 'PAD', 'Dana_Transfer']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for ax, col in zip(axes, cols_to_plot):
    data = df[col].dropna()
    ax.boxplot(data / 1e9, patch_artist=True,
               boxprops=dict(facecolor='#3498db', alpha=0.7),
               medianprops=dict(color='red', linewidth=2),
               flierprops=dict(marker='o', color='red', alpha=0.5))
    ax.set_title(col.replace('_', '\n'), fontsize=9, fontweight='bold')
    ax.set_ylabel('Miliar Rp')
    ax.set_xticks([])

plt.suptitle('Boxplot: Deteksi Outlier pada Kolom Utama\n(titik merah = outlier)', 
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 4.2 Hitung persentase realisasi & visualisasi distribusinya
df_clean_viz = df[(df['Anggaran_APBD'] > 0) & (df['Realisasi_APBD'].notna())].copy()
df_clean_viz['Pct_Realisasi'] = df_clean_viz['Realisasi_APBD'] / df_clean_viz['Anggaran_APBD'] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_clean_viz['Pct_Realisasi'].clip(-50, 200), bins=25,
             color='#9b59b6', edgecolor='white', linewidth=1.2)
axes[0].axvline(60, color='orange', linestyle='--', linewidth=1.5, label='60% (batas Cukup)')
axes[0].axvline(80, color='green', linestyle='--', linewidth=1.5, label='80% (batas Baik)')
axes[0].axvline(90, color='darkgreen', linestyle='--', linewidth=1.5, label='90% (batas Sangat Baik)')
axes[0].set_title('Distribusi % Realisasi APBD', fontweight='bold')
axes[0].set_xlabel('% Realisasi')
axes[0].set_ylabel('Frekuensi')
axes[0].legend(fontsize=8)

# Boxplot per predikat
df_box = df_clean_viz.dropna(subset=['Predikat'])
order = [p for p in ['Kurang', 'Cukup', 'Baik', 'Sangat Baik'] if p in df_box['Predikat'].unique()]
palette = {'Kurang': '#e74c3c', 'Cukup': '#f39c12', 'Baik': '#2ecc71', 'Sangat Baik': '#27ae60'}
sns.boxplot(data=df_box, x='Predikat', y='Pct_Realisasi', order=order, palette=palette, ax=axes[1])
axes[1].set_title('% Realisasi per Predikat Kinerja', fontweight='bold')
axes[1].set_xlabel('Predikat')
axes[1].set_ylabel('% Realisasi')

plt.tight_layout()
plt.show()

## 🧹 5. Phase 3: Data Cleaning

In [ ]:
# 5.1 Tangani Missing Values
df_cleaned = df.copy()

print('SEBELUM penanganan missing values:')
print(f'  Jumlah missing: {df_cleaned.isnull().sum().sum()}')

# Imputasi kolom numerik dengan median (robust terhadap outlier)
for col in df_cleaned.select_dtypes(include=[np.number]).columns:
    if df_cleaned[col].isnull().sum() > 0:
        median_val = df_cleaned[df_cleaned[col] > 0][col].median()  # Median dari nilai positif saja
        n_filled = df_cleaned[col].isnull().sum()
        df_cleaned[col].fillna(median_val, inplace=True)
        print(f'  ✅ {col}: {n_filled} nilai diisi dengan median = {median_val:,.0f}')

print(f'\nSESUDAH penanganan missing values:')
print(f'  Jumlah missing: {df_cleaned.isnull().sum().sum()}')

In [ ]:
# 5.2 Tangani Nilai Negatif — Tandai sebagai outlier/error
print('PENANGANAN NILAI NEGATIF (Anggaran_APBD)')
print('=' * 50)

neg_mask = df_cleaned['Anggaran_APBD'] < 0
print(f'Jumlah baris dengan Anggaran negatif: {neg_mask.sum()}')
print('Baris tersebut:')
print(df_cleaned[neg_mask][['Kode_Pemda', 'Tahun', 'Anggaran_APBD', 'Realisasi_APBD', 'Predikat']])

# Pilihan 1: Hapus baris dengan anggaran negatif (kemungkinan error input)
df_cleaned = df_cleaned[~neg_mask].copy()
print(f'\n✅ {neg_mask.sum()} baris dengan anggaran negatif dihapus')
print(f'Dataset sekarang: {df_cleaned.shape[0]} baris')

In [ ]:
# 5.3 Deteksi & Tangani Outlier menggunakan IQR
print('DETEKSI OUTLIER — METODE IQR')
print('=' * 55)

col_outlier = 'Anggaran_APBD'

Q1 = df_cleaned[col_outlier].quantile(0.25)
Q3 = df_cleaned[col_outlier].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_cleaned[(df_cleaned[col_outlier] < lower_bound) | (df_cleaned[col_outlier] > upper_bound)]

print(f'Kolom : {col_outlier}')
print(f'Q1    : Rp {Q1:,.0f}')
print(f'Q3    : Rp {Q3:,.0f}')
print(f'IQR   : Rp {IQR:,.0f}')
print(f'Batas Bawah : Rp {lower_bound:,.0f}')
print(f'Batas Atas  : Rp {upper_bound:,.0f}')
print(f'\nJumlah outlier terdeteksi: {len(outliers)}')

if len(outliers) > 0:
    print(outliers[['Kode_Pemda', col_outlier]].to_string())

In [ ]:
# 5.4 Rekalkukasi Predikat berdasarkan aturan bisnis
print('VALIDASI & REKALKUKASI PREDIKAT')
print('=' * 50)

def hitung_predikat(row):
    """Hitung predikat berdasarkan persentase realisasi."""
    if pd.isna(row['Anggaran_APBD']) or row['Anggaran_APBD'] <= 0:
        return None
    pct = row['Realisasi_APBD'] / row['Anggaran_APBD'] * 100
    if pct >= 90:
        return 'Sangat Baik'
    elif pct >= 80:
        return 'Baik'
    elif pct >= 60:
        return 'Cukup'
    else:
        return 'Kurang'

df_cleaned['Predikat_Rekalk'] = df_cleaned.apply(hitung_predikat, axis=1)

# Cek inkonsistensi
inkonsisten = df_cleaned[df_cleaned['Predikat'] != df_cleaned['Predikat_Rekalk']]
print(f'Jumlah inkonsistensi: {len(inkonsisten)}')
if len(inkonsisten) > 0:
    print(inkonsisten[['Kode_Pemda', 'Predikat', 'Predikat_Rekalk']].head(10))

# Update Predikat
df_cleaned['Predikat'] = df_cleaned['Predikat_Rekalk']
df_cleaned.drop(columns=['Predikat_Rekalk'], inplace=True)
print('\n✅ Predikat telah direkalkukasi')

## 🔄 6. Phase 4: Data Transformation

In [ ]:
# 6.1 Feature Engineering — buat fitur baru yang informatif
df_transformed = df_cleaned.copy()

df_transformed['Pct_Realisasi'] = (
    df_transformed['Realisasi_APBD'] / df_transformed['Anggaran_APBD'] * 100
).round(2)

df_transformed['Rasio_Kemandirian'] = (
    df_transformed['PAD'] / df_transformed['Anggaran_APBD'] * 100
).round(2)

df_transformed['Rasio_Belanja_Pegawai'] = (
    df_transformed['Belanja_Pegawai'] / df_transformed['Realisasi_APBD'] * 100
).round(2)

df_transformed['Sisa_Anggaran'] = (
    df_transformed['Anggaran_APBD'] - df_transformed['Realisasi_APBD']
)

print('Feature baru yang dibuat:')
new_features = ['Pct_Realisasi', 'Rasio_Kemandirian', 'Rasio_Belanja_Pegawai', 'Sisa_Anggaran']
print(df_transformed[new_features].describe().round(2))

In [ ]:
# 6.2 Encoding Variabel Kategorikal (Predikat)
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# Label Encoding untuk Predikat (Ordinal)
predikat_map = {'Kurang': 0, 'Cukup': 1, 'Baik': 2, 'Sangat Baik': 3}
df_transformed['Predikat_Encoded'] = df_transformed['Predikat'].map(predikat_map)

print('Label Encoding Predikat:')
print(predikat_map)
print()
print(df_transformed[['Predikat', 'Predikat_Encoded']].value_counts().sort_index())

In [ ]:
# 6.3 Normalisasi (Min-Max Scaling)
feature_cols = ['Anggaran_APBD', 'Realisasi_APBD', 'PAD',
                'Dana_Transfer', 'Belanja_Pegawai', 'Pct_Realisasi',
                'Rasio_Kemandirian']

scaler = MinMaxScaler()
df_normalized = df_transformed.copy()
df_normalized[feature_cols] = scaler.fit_transform(df_transformed[feature_cols].fillna(0))

print('Setelah Min-Max Normalization (rentang 0–1):')
df_normalized[feature_cols].describe().round(3)

## ✅ 7. Ringkasan Preprocessing

In [ ]:
# Ringkasan sebelum & sesudah preprocessing
print('=' * 55)
print('RINGKASAN PREPROCESSING')
print('=' * 55)
print(f'  Baris Awal          : {df_original.shape[0]}')
print(f'  Baris Sesudah Cleaning: {df_cleaned.shape[0]}')
print(f'  Baris Dihapus       : {df_original.shape[0] - df_cleaned.shape[0]}')
print()
print(f'  Missing Values Awal  : {df_original.isnull().sum().sum()}')
print(f'  Missing Values Akhir : {df_cleaned.isnull().sum().sum()}')
print()
print(f'  Kolom Awal   : {df_original.shape[1]}')
print(f'  Kolom Akhir  : {df_transformed.shape[1]}')
print(f'  Kolom Baru   : {df_transformed.shape[1] - df_original.shape[1]} (feature engineering + encoding)')

# Simpan hasil cleaned
df_transformed.to_csv('keuangan_pemda_cleaned.csv', index=False)
print('\n✅ Dataset bersih disimpan: keuangan_pemda_cleaned.csv')

## 📝 8. Latihan

1. **Imputasi alternatif:** Coba isi missing values dengan **mean** bukan median. Apa perbedaan hasilnya? Mana yang lebih baik untuk data ini?
2. **Outlier handling:** Selain menghapus, coba **cap** nilai outlier dengan batas IQR. Bandingkan distribusinya sebelum dan sesudah.
3. **Feature engineering baru:** Buat fitur `Rasio_Belanja_Modal` = Belanja_Modal / Total Belanja. Apakah fitur ini berguna?
4. **Encoding alternatif:** Coba **One-Hot Encoding** untuk kolom Predikat. Kapan One-Hot lebih baik dari Label Encoding?
5. **Standardisasi:** Bandingkan MinMaxScaler dengan StandardScaler. Kapan masing-masing lebih tepat digunakan?

In [ ]:
# ✏️ Ruang Eksplorasi Mandiri
# Kerjakan latihan di sini!

# Contoh starter untuk latihan 1:
# mean_val = df[col].mean()
# df[col].fillna(mean_val, inplace=True)